# MASLD-LLM C3 Evaluations

Self-contained Colab notebook for evaluating 1.7B model on C3 corpora.
Downloads data directly from GitHub and runs evaluations in Colab.

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch==2.14.1 transformers accelerate peft pandas numpy huggingface_hub

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2: Download corpora from GitHub
import urllib.request
import gzip
import json
import os

GITHUB_BASE = 'https://raw.githubusercontent.com/subhadipbanerjee/masld-llm/main/prompts'

corpora = {
    'MB-85xx': 'MB-85xx_progression_test.jsonl.gz',
    'MB-87xx': 'MB-87xx_liverplasma_test.jsonl.gz'
}

for name, filename in corpora.items():
    url = f'{GITHUB_BASE}/{filename}'
    output = f'/content/{filename}'
    print(f'Downloading {name}...')
    urllib.request.urlretrieve(url, output)
    
    # Load and count records
    records = []
    with gzip.open(output, 'rt') as f:
        for line in f:
            if line.strip().startswith('{"__meta__"'):
                continue
            records.append(json.loads(line))
    print(f'  {name}: {len(records)} records')

print('All corpora downloaded')

In [ ]:
# Cell 3: Load 1.7B model
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import gc

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

base_model = 'Qwen/Qwen3-1.7B'
adapter_path = 'subhadipbanerjee/qwen3-1.7b-masld-sft'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(base_model)

print('Loading base model...')
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)

print('Loading adapter...')
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

print('Model loaded successfully')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 4: Evaluate MB-85xx
def evaluate_corpus(corpus_path, output_path, model, tokenizer):
    import json
    import gzip
    
    # Load corpus
    records = []
    with gzip.open(corpus_path, 'rt') as f:
        for line in f:
            if line.strip().startswith('{"__meta__"'):
                continue
            records.append(json.loads(line))
    
    print(f'Loaded {len(records)} records')
    
    # Generation config
    gen_config = {
        'max_new_tokens': 512,
        'do_sample': False,
        'pad_token_id': tokenizer.eos_token_id,
    }
    
    # Evaluate
    results = []
    for i, rec in enumerate(records):
        prompt = rec['prompt']
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_config)
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer = response[len(prompt):].strip()
        
        results.append({**rec, 'response': response, 'answer': answer})
        
        if (i + 1) % 10 == 0:
            print(f'  {i + 1}/{len(records)}')
    
    # Save results
    with gzip.open(output_path, 'wt') as f:
        f.write(json.dumps({'__meta__': {'n_records': len(results), 'model': 'qwen3-1.7b-masld'}}, indent=1) + '\n')
        for r in results:
            f.write(json.dumps(r) + '\n')
    
    return len(results)

# Run evaluation
output = evaluate_corpus(
    '/content/MB-85xx_progression_test.jsonl.gz',
    '/content/MB-85xx_eval.jsonl.gz',
    model, tokenizer
)
print(f'MB-85xx evaluation complete: {output} records')

# Free GPU memory
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Cell 5: Evaluate MB-87xx
output = evaluate_corpus(
    '/content/MB-87xx_liverplasma_test.jsonl.gz',
    '/content/MB-87xx_eval.jsonl.gz',
    model, tokenizer
)
print(f'MB-87xx evaluation complete: {output} records')

# Free GPU memory
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Cell 6: Verify outputs
import os

for fname in ['MB-85xx_eval.jsonl.gz', 'MB-87xx_eval.jsonl.gz']:
    fpath = f'/content/{fname}'
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        with gzip.open(fpath, 'rt') as f:
            lines = len(f.readlines())
        print(f'{fname}: {lines} lines, {size} bytes')
    else:
        print(f'{fname}: NOT FOUND')

In [ ]:
# Cell 7: Download outputs to local
from google.colab import files

print('Downloading evaluation results...')
files.download('/content/MB-85xx_eval.jsonl.gz')
files.download('/content/MB-87xx_eval.jsonl.gz')

print('Download complete. Upload to masld-llm/results/frontier_runs/')